In [2]:
import pandas as pd

file_path = "/lakehouse/default/Files/raw/Online Retail.xlsx"

data = pd.read_excel(file_path)

from datetime import datetime, timezone

data['SourceFile'] = "Online Retail.xlsx"
data['IngestedAt'] = datetime.now(timezone.utc)

data["IngestedAt"] = data["IngestedAt"].dt.tz_localize(None)

print("Rows:", data.shape[0])
print("Columns:", data.shape[1])

data.head()

data.dtypes

StatementMeta(, f94b45c9-6ada-4dae-b74b-24b0b62e981d, 5, Finished, Available, Finished, False)

Rows: 541909
Columns: 10


InvoiceNo              object
StockCode              object
Description            object
Quantity                int64
InvoiceDate    datetime64[ns]
UnitPrice             float64
CustomerID            float64
Country                object
SourceFile             object
IngestedAt     datetime64[us]
dtype: object

In [3]:
bronze_df = spark.createDataFrame(data)

print("Rows:", bronze_df.count())
print("Columns:", len(bronze_df.columns))

bronze_df.printSchema()

StatementMeta(, f94b45c9-6ada-4dae-b74b-24b0b62e981d, 6, Finished, Available, Finished, False)

/opt/spark/python/lib/pyspark.zip/pyspark/sql/pandas/conversion.py:416: UserWarning: createDataFrame attempted Arrow optimization because 'spark.sql.execution.arrow.pyspark.enabled' is set to true; however, failed by the reason below:
  Could not convert 'C536379' with type str: tried to convert to int64
Attempting non-optimization as 'spark.sql.execution.arrow.pyspark.fallback.enabled' is set to true.


Rows: 541909
Columns: 10
root
 |-- InvoiceNo: string (nullable = true)
 |-- StockCode: string (nullable = true)
 |-- Description: string (nullable = true)
 |-- Quantity: long (nullable = true)
 |-- InvoiceDate: timestamp (nullable = true)
 |-- UnitPrice: double (nullable = true)
 |-- CustomerID: double (nullable = true)
 |-- Country: string (nullable = true)
 |-- SourceFile: string (nullable = true)
 |-- IngestedAt: timestamp (nullable = true)



In [4]:
# Create Bronze schema if it does not already exist
spark.sql("CREATE SCHEMA IF NOT EXISTS bronze")

# Write the source-aligned data as a managed Delta table
(
    bronze_df.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("bronze.raw_sales")
)

print("Bronze table created successfully.")

StatementMeta(, f94b45c9-6ada-4dae-b74b-24b0b62e981d, 7, Finished, Available, Finished, False)

Bronze table created successfully.


In [5]:
bronze_check = spark.table("bronze.raw_sales")

print("Rows:", bronze_check.count())
print("Columns:", len(bronze_check.columns))

bronze_check.printSchema()

StatementMeta(, f94b45c9-6ada-4dae-b74b-24b0b62e981d, 8, Finished, Available, Finished, False)

Rows: 541909
Columns: 10
root
 |-- InvoiceNo: string (nullable = true)
 |-- StockCode: string (nullable = true)
 |-- Description: string (nullable = true)
 |-- Quantity: long (nullable = true)
 |-- InvoiceDate: timestamp (nullable = true)
 |-- UnitPrice: double (nullable = true)
 |-- CustomerID: double (nullable = true)
 |-- Country: string (nullable = true)
 |-- SourceFile: string (nullable = true)
 |-- IngestedAt: timestamp (nullable = true)

